In [1]:
import random
import re
import os
import sys
from pathlib import Path
from moviepy.video.VideoClip import ImageClip
from moviepy import VideoFileClip, AudioFileClip, concatenate_videoclips, CompositeVideoClip
from pocket_tts import TTSModel
import scipy.io.wavfile
from scipy.io import wavfile
import wave
import contextlib
import soundfile as sf
import numpy as np
from datetime import timedelta
from moviepy.video.tools.subtitles import SubtitlesClip
from moviepy.video.VideoClip import TextClip
import unicodedata
from moviepy.audio.io.AudioFileClip import AudioFileClip

from sqlalchemy.orm import Session
from sqlalchemy import text

import torch
import open_clip

c:\Users\georg\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
root = Path.cwd()
while not (root / "src").exists():
    root = root.parent

sys.path.append(str(root))
from src.db.session import engine

In [3]:
script_path = r"C:\Uni\GP\ECHO\notebooks\video_generation\video_generation_landmarks\scripts\Temple of Hatshepsut in Deir El Bahari.txt"
with open(script_path, "r", encoding="utf-8") as file:
    script = file.read()

sections = [section.strip() for section in script.split(".") if section.strip()]
sections = [s + "." if not s.endswith(".") else s for s in sections]

for i, section in enumerate(sections, 1):
    print(f"Section {i}:")
    print(section)
    print("-" * 40)

Section 1:
Nestled against the towering cliffs of Deir el-Bahari lies the Mortuary Temple of Hatshepsut, a masterpiece of ancient Egyptian architecture.
----------------------------------------
Section 2:
Known as the "Holy of Holies," this limestone monument was commissioned by the visionary Queen-Pharaoh Hatshepsut around 1479 BC.
----------------------------------------
Section 3:
Designed by her steward Senenmut, the temple features three massive terraces connected by grand ramps, harmonizing perfectly with the rugged landscape.
----------------------------------------
Section 4:
The temple’s intricate reliefs immortalize a prosperous reign.
----------------------------------------
Section 5:
The Punt Colonnade depicts a legendary trade expedition to the "Land of the Gods," while the Birth Colonnade asserts Hatshepsut’s divine right to rule as the daughter of Amun-Ra.
----------------------------------------
Section 6:
Although her successor, Thutmose III, later attempted to erase 

In [10]:
#Text-to-Speech
tts_model = TTSModel.load_model()
voice_state = tts_model.get_state_for_audio_prompt("alba")

i = 0
for p in sections:
    audio = tts_model.generate_audio(voice_state, p)
    scipy.io.wavfile.write(f"C:\\Uni\\GP\\ECHO\\notebooks\\video_generation\\Outputs\\audio_{i}.wav", tts_model.sample_rate, audio.numpy())
    i += 1

In [11]:
wav_files = sorted([f for f in os.listdir("C:\\Uni\\GP\\ECHO\\notebooks\\video_generation\\Outputs") if f.endswith('.wav')])
print(wav_files)
wav_files = [Path("C:\\Uni\\GP\\ECHO\\notebooks\\video_generation\\Outputs") / f for f in wav_files]
audio_data = []
samplerate = None

for file_name in wav_files:
    data, sr = sf.read(file_name)
    
    if samplerate is None:
        samplerate = sr
    elif sr != samplerate:
        raise ValueError("Sample rates do not match!")

    audio_data.append(data)

# Concatenate along time axis
combined = np.concatenate(audio_data, axis=0)

# Write output (keeps float format safely)
sf.write("C:\\Uni\\GP\\ECHO\\notebooks\\video_generation\\Outputs\\audio_combined.wav", combined, samplerate)

['audio_0.wav', 'audio_1.wav', 'audio_2.wav', 'audio_3.wav', 'audio_4.wav', 'audio_5.wav', 'audio_6.wav', 'audio_combined.wav']


In [12]:
durations = []
images_needed = []
seconds = []
for i in range(len(sections)):
    file_path = f"C:\\Uni\\GP\\ECHO\\notebooks\\video_generation\\Outputs\\audio_{i}.wav"
    # Returns the sample rate (Fs) and the data as a NumPy array
    Fs, data = wavfile.read(file_path) 
    # The length of the data array divided by the sample rate gives the duration
    duration_seconds = len(data) / float(Fs)
    durations.append(duration_seconds)
    images_needed.append(max(1, int(duration_seconds / 6)))
    section_seconds = duration_seconds / images_needed[-1]
    for img in range(images_needed[-1]):
        seconds.append(section_seconds)
    os.remove(file_path)

print(f"Durations of audio files: {durations}")
print(f"Images needed for each paragraph: {images_needed}")
print(f"Seconds for each image: {seconds}")

Durations of audio files: [7.68, 8.4, 7.68, 4.4, 9.2, 7.44, 12.08]
Images needed for each paragraph: [1, 1, 1, 1, 1, 1, 2]
Seconds for each image: [7.68, 8.4, 7.68, 4.4, 9.2, 7.44, 6.04, 6.04]


In [ ]:
name = Path(script_path).stem
with engine.connect() as conn:
    result = conn.execute(text("SELECT image_embedding FROM landmark_images as li JOIN landmarks as l ON li.landmark_id = l.id where l.name = :name"), {"name": name})

    landmark_images = result.fetchall()

landmark_images

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained="laion2b_s32b_b79k"
)
model = model.to(device)
model.eval()

c:\Users\georg\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\georg\.cache\huggingface\hub\models--laion--CLIP-ViT-H-14-laion2B-s32B-b79K. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [ ]:
for i, section in enumerate(sections, 1):
    
    text_tokens = open_clip.get_tokenizer()([section]).to(device)
    
    with torch.no_grad():
        
        text_embedding = model.encode_text(text_tokens)
        text_embedding /= text_embedding.norm(dim=-1, keepdim=True)

    embedding_list = text_embedding.cpu().numpy().tolist()[0]

    metadata_filter = {"pharaoh_name": "Rameses II"}

    '''    results = collection.query(
    query_embeddings=[embedding_list],
    n_results=images_needed[i],
    include=["documents", "metadatas"],
    where=metadata_filter
    )'''
    print(f"Section {i} top {images_needed[i]} matches:")
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        print("Metadata:", meta)
        try:
            img = Image.open(doc)  # doc is the path to the image
            display(img)  # displays the image
        except Exception as e:
            print("Failed to open image:", doc, "Error:", e)
    print("-" * 50)